# Part 3A: CAM_FRONT 3D-to-2D annotation projection

This notebook converts nuScenes `v1.0-mini` global 3D sample annotations into 2D CAM_FRONT boxes and displays overlays on the original camera images. It does not split data, train a model, install PyTorch, copy source images, or modify the metadata database.

The package uses the nuScenes `Box`, quaternion, and camera projection utilities. It transforms global coordinates to ego coordinates and then to camera coordinates using inverse ego and calibrated-sensor poses. Cuboid edges are clipped at a positive near plane before projection; projected boxes are clipped to image bounds and invalid/unsupported annotations are retained in the rejection report. Visibility is recorded, not used as an occlusion filter. The timestamp varies per conversion; ordered records and exports are deterministic for the same input/configuration.

In [1]:
from pathlib import Path
import os

from sqlalchemy import inspect, select
from sqlalchemy.orm import Session

from scenarioforge.cli import load_settings
from scenarioforge.database.connection import create_database_engine
from scenarioforge.database.models import Dataset
from scenarioforge.datasets.nuscenes_adapter import NuScenesAdapter
from scenarioforge.vision.config import ProjectionConfig
from scenarioforge.vision.converter import convert_camera_samples
from scenarioforge.vision.dataset_builder import write_conversion
from scenarioforge.vision.inspection import render_inspection

settings = load_settings()
dataset_version = settings.dataset_version
configured_root = Path(os.environ["NUSCENES_ROOT"]).expanduser() if os.environ.get("NUSCENES_ROOT") else settings.dataset_root
if configured_root.is_dir():
    dataset_root = configured_root.resolve()
else:
    engine = create_database_engine(settings)
    try:
        if not inspect(engine).has_table(Dataset.__tablename__):
            raise FileNotFoundError("Configure NUSCENES_ROOT or ingest the dataset root into the metadata database first.")
        with Session(engine) as session:
            stored_root = session.scalar(select(Dataset.root_path).where(Dataset.source_id == f"nuscenes:{dataset_version}"))
    finally:
        engine.dispose()
    if not stored_root or not Path(stored_root).is_dir():
        raise FileNotFoundError("No valid nuScenes root found; set NUSCENES_ROOT.")
    dataset_root = Path(stored_root).resolve()

output_dir = settings.project_root / "data" / "processed" / "vision"
print(f"Dataset root: {dataset_root.name}")
print(f"Output directory: {output_dir.relative_to(settings.project_root)}")

Dataset root: downloaded-dataset
Output directory: data\processed\vision


## Load source metadata and convert

The default policy uses CAM_FRONT, a 0.1 m near plane, at least three projected points, a minimum 1 px box area, and a maximum 98% image-area box. The default mapping is the explicit ten-class nuScenes detection mapping. No visibility-level filtering is applied.

In [2]:
adapter = NuScenesAdapter(dataset_root, dataset_version)
problems = adapter.validate_accessibility()
if problems:
    raise RuntimeError("Dataset accessibility check failed:\n" + "\n".join(problems))

config = ProjectionConfig()
source_samples = list(adapter.iter_camera_samples(config.camera_channel))
result = convert_camera_samples(
    source_samples,
    dataset_root=dataset_root,
    dataset_version=dataset_version,
    visibility_levels=adapter.visibility_levels,
    config=config,
    processing_timestamp="2026-01-01T00:00:00+00:00",
)
files = write_conversion(result, output_dir)
print(f"Samples: {len(source_samples):,}")
print(f"Images: {len(result.images):,}")
print(f"Accepted annotations: {len(result.accepted_annotations):,}")
print(f"Rejected annotations: {len(result.rejected_annotations):,}")
print(f"Export files: {sorted(path.name for path in files.values())}")

Samples: 404
Images: 404
Accepted annotations: 5,325
Rejected annotations: 13,213
Export files: ['annotations.csv', 'annotations.json', 'category_mapping.json', 'conversion_report.json', 'dataset_manifest.json', 'image_manifest.csv', 'rejected_annotations.csv']


## Conversion report and consistency checks

Rejected annotations have explicit reasons such as `object_behind_camera`, `outside_image`, and `unsupported_category`. These counts describe the source labels and projection policy; they are not detector metrics.

In [3]:
report = result.report
print("Category counts:", report["category_distribution"])
print("Rejection counts:", report["rejected_annotations_by_reason"])
print("Rejections by reason and mapped category:", report["rejected_annotations_by_reason_and_category"])
print("Image statuses:", report["image_status_counts"])
print("Sanity checks:", report["sanity_checks"])

assert report["total_source_annotations_considered"] == len(result.accepted_annotations) + len(result.rejected_annotations)
assert report["total_camera_images_processed"] == len(result.images)
assert report["sanity_checks"]["accepted_boxes_out_of_image_bounds"] == 0
assert report["sanity_checks"]["accepted_zero_or_negative_area_boxes"] == 0

Category counts: {'barrier': 527, 'bicycle': 82, 'bus': 182, 'car': 2351, 'construction_vehicle': 101, 'motorcycle': 147, 'pedestrian': 1134, 'traffic_cone': 594, 'trailer': 17, 'truck': 190}
Rejection counts: {'object_behind_camera': 8682, 'outside_image': 4357, 'unsupported_category': 174}
Rejections by reason and mapped category: {'object_behind_camera': {'barrier': 1498, 'bicycle': 104, 'bus': 126, 'car': 3256, 'construction_vehicle': 75, 'motorcycle': 205, 'pedestrian': 2530, 'traffic_cone': 518, 'trailer': 41, 'truck': 329}, 'outside_image': {'barrier': 298, 'bicycle': 57, 'bus': 102, 'car': 2012, 'construction_vehicle': 20, 'motorcycle': 119, 'pedestrian': 1351, 'traffic_cone': 266, 'trailer': 2, 'truck': 130}, 'unsupported_category': {'unmapped': 174}}
Image statuses: {'ok': 404}
Sanity checks: {'accepted_boxes_out_of_image_bounds': 0, 'accepted_zero_or_negative_area_boxes': 0, 'accepted_annotations_with_unexpected_class_ids': 0, 'duplicate_source_annotation_ids': 0, 'images_wi

## Inspect actual front-camera images

Each view uses source images from the dataset root. Labels show the mapped category and a source annotation token prefix; the bottom caption summarizes rejected annotation reasons for that image. Review examples from different scenes and classes before considering the annotations for downstream use.

In [4]:
from IPython.display import Image as NotebookImage, display

inspection_dir = output_dir / "inspection" / "notebook"
inspection_examples = {}
for mode in ("many", "partial", "empty", "rejected"):
    paths = render_inspection(
        files["annotations_json"],
        dataset_root,
        inspection_dir / mode,
        mode=mode,
        count=1,
        seed=42,
    )
    inspection_examples[mode] = paths[0]
    print(f"{mode}: {paths[0].relative_to(output_dir)}")
    display(NotebookImage(filename=str(paths[0])))

many: inspection\notebook\many\01_4d7cf1ad5f9c4842bb6f135e058deed4.png


partial: inspection\notebook\partial\01_3ff58ee0fc4748d9895300f02114aa56.png


empty: inspection\notebook\empty\01_b4ff30109dd14c89b24789dc5713cf8c.png


rejected: inspection\notebook\rejected\01_b6c420c3a5bd4a219b1cb82ee5ea0aa7.png


## Limitations and next step

A 3D cuboid's image-space extent is a geometric projection, not a human-drawn visible-object boundary. Small/occluded objects can have imperfect projected bounds; source visibility is only a coarse source attribute. The mini release is useful for validating this pipeline, not for statistically strong generalization claims. This notebook does not create splits or begin Part 3B model training; the next step is manual review of the exported report and overlays.